<a href="https://colab.research.google.com/github/JediricX/desercionIA/blob/main/Alerta_temprana_abandono_universitario_3_Entrenamiento_modelo_Random_Forest_y_explicabilidad_SHA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Modelo IA para la detección temprana del abandono universitario basado en dataset OULAD

Objetivo del modelo:

Diseñar e implementar un Sistema de Alerta Temprana basado en Inteligencia Artificial que permita predecir el riesgo de abandono en programas universitarios a distancia, mediante la comparación de algoritmos de clasificación supervisada (Random Forest, Árboles de Decisión, XGBoost y Regresión Logística), con el fin de ofrecer a las instituciones educativas una herramienta preventiva para la retención estudiantil.

Autores:


*   Richard Humberto Campos Ballesteros
*   José Andía



# **1. Importación de librerías**

In [ ]:
# Manejo de datos
import pandas as pd
import numpy as np
import os

# Preprocesamiento
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Métricas
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.metrics import RocCurveDisplay

# Desbalance de clases
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


# **2. Carga de datos generados en la fase de Ingeniería de características**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data_path = '/content/drive/MyDrive/Colab Notebooks/Modelos/Data/processed_student_data'

X_train_resampled = np.load(os.path.join(data_path, 'X_train_resampled.npy'))
y_train_resampled = np.load(os.path.join(data_path, 'y_train_resampled.npy'))
X_test = np.load(os.path.join(data_path, 'X_test.npy'))
y_test = np.load(os.path.join(data_path, 'y_test.npy'))

processed_data_full = {
    'X_train_resampled': X_train_resampled,
    'y_train_resampled': y_train_resampled,
    'X_test': X_test,
    'y_test': y_test
}

print("Data loaded successfully into processed_data_full!")
print(f"X_train_resampled shape: {X_train_resampled.shape}")
print(f"y_train_resampled shape: {y_train_resampled.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

# **3. Configuración modelo Random Forest con hyperparámetros**

In [ ]:
rf_model = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    n_estimators=400,
    min_samples_split=5,
    min_samples_leaf=1,
    max_features=None,
    max_depth=None,
    bootstrap=True
)

print("Training RandomForestClassifier...")
rf_model.fit(X_train_resampled, y_train_resampled)
print("RandomForestClassifier training complete!")

## 4. Evaluación del Modelo Random Forest

In [ ]:
print("Making predictions on the test set...")
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

print("Generating Classification Report...")
print(classification_report(y_test, y_pred))

print("Generating Confusion Matrix...")
print(confusion_matrix(y_test, y_pred))

print("Calculating ROC AUC Score...")
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC AUC Score: {roc_auc:.4f}")

In [ ]:
# Plot ROC Curve
fig, ax = plt.subplots(figsize=(8, 6))
roc_display = RocCurveDisplay.from_estimator(rf_model, X_test, y_test, ax=ax)
plt.title('Curva ROC - Random Forest Classifier')
plt.grid(True)
plt.show()

## 5. Importancia de las Variables (Feature Importance)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Obtener la importancia de las características del modelo
feature_importances = rf_model.feature_importances_

# Crear nombres genéricos para las características si no se tienen los originales
# Asumiendo que X_train_resampled es un array numpy y no tiene nombres de columnas
num_features = X_train_resampled.shape[1]
feature_names = [f'Feature {i+1}' for i in range(num_features)]

# Crear un DataFrame para facilitar la visualización
importances_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})

# Ordenar por importancia de forma descendente
importances_df = importances_df.sort_values(by='Importance', ascending=False)

# Visualizar la importancia de las características
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importances_df)
plt.title('Importancia de las Características (Random Forest)')
plt.xlabel('Importancia')
plt.ylabel('Característica')
plt.tight_layout()
plt.show()

## 6. Validación Cruzada (Cross-Validation)

In [ ]:
from sklearn.model_selection import cross_val_score

# Configurar la validación cruzada estratificada
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Performing cross-validation...")
# Realizar la validación cruzada y obtener los puntajes ROC AUC para cada fold
cv_scores = cross_val_score(rf_model, X_train_resampled, y_train_resampled, cv=skf, scoring='roc_auc', n_jobs=-1)

print(f"ROC AUC scores for each fold: {cv_scores}")
print(f"Mean ROC AUC: {cv_scores.mean():.4f}")
print(f"Standard deviation of ROC AUC: {cv_scores.std():.4f}")


## 7. Reporte de Clasificación Detallado y Análisis

In [ ]:
print("Generando Reporte de Clasificación detallado para el conjunto de prueba:")
y_pred = rf_model.predict(X_test)
print(classification_report(y_test, y_pred))

## 8. Visualización de la Curva ROC para Capacidad Discriminativa

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

# Ensure y_pred_proba is available, if not, re-calculate
if 'y_pred_proba' not in locals():
    y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(8, 6))
roc_display = RocCurveDisplay.from_estimator(rf_model, X_test, y_test, ax=ax)
plt.title('Curva ROC - Random Forest Classifier (Evaluación Discriminativa)')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.grid(True)
plt.show()

## 9. Explicabilidad del Modelo con SHAP

In [ ]:
import shap

print("Generating SHAP explanations...")

# Crear un explicador SHAP para el modelo Random Forest
# Use 'model_output="predict_proba"' for classification to explain probability of the positive class
explainer = shap.TreeExplainer(rf_model, model_output="predict_proba")

# Calcular los valores SHAP para el conjunto de prueba
shap_values = explainer.shap_values(X_test)

# shap_values will be a list of arrays for multi-class, for binary classification we take the second array (class 1)
# If it's a list, it usually contains two arrays: shap_values[0] for class 0, shap_values[1] for class 1
shap_values_class_1 = shap_values[1]

# Asumimos que no tenemos los nombres de las características, por lo que usaremos nombres genéricos
num_features = X_test.shape[1]
feature_names = [f'Feature {i+1}' for i in range(num_features)]

# Visualizar la importancia de las características con SHAP summary plot
print("Plotting SHAP summary plot...")
shap.summary_plot(shap_values_class_1, X_test, feature_names=feature_names, plot_type="bar")
plt.title('SHAP Feature Importance (Clase 1 - Abandono)')
plt.show()

# También podemos usar un summary plot más detallado (beeswarm plot)
print("Plotting SHAP beeswarm plot...")
shap.summary_plot(shap_values_class_1, X_test, feature_names=feature_names)
plt.title('SHAP Summary Plot (Clase 1 - Abandono)')
plt.show()

## 10. Guardar el Modelo Entrenado

In [ ]:
import joblib

# Definir la ruta para guardar el modelo
model_save_path = os.path.join(data_path, 'random_forest_model.joblib')

print(f"Saving the trained Random Forest model to: {model_save_path}")
joblib.dump(rf_model, model_save_path)
print("Model saved successfully!")